# Lesson 3 — From raw files to clean data tables

**Course:** Python from zero for Materials Science PhD students  
**Session:** Hands-on 2  
**Nominal duration:** ~3 hours (about 2 h of core material + optional exercises)

## The story for today

Imagine a colleague just sent you a data file straight from the **tensile-testing machine**. It holds measurements for **5 metal specimens** (`S01`–`S05`), each given a different **heat treatment** (annealed, quenched, aged, as-received).

Everyone in the lab wants the answer to one question:

> **Which heat treatment gives the strongest material?**

But the file is a *raw machine export*: hundreds of rows, with a few sensor glitches hiding inside. Our job today is to go from that messy file to a clear answer:

```text
raw file → read it → look at it → compute stress → remove glitches → one number per specimen → the winner
```

Along the way we learn the everyday `pandas` moves that show up in almost any data analysis. The goal is **not** to master pandas today — just to get comfortable with the basics.

> **Keep in mind:** a good analysis notebook is *repeatable* — run it top to bottom and it always produces the same clean data and the same answer.

## 0. Setup

We will use a few standard libraries:

- `pathlib`, from the Python standard library, to handle file paths;
- `pandas`, to read and manipulate tables;
- `matplotlib`, to make a very simple check plot at the end;
- `math`, for constants such as `math.pi` (we met this in the previous lesson).

Today, `pandas` is the main tool.

Conventionally, pandas is imported as `pd`.

In [ ]:
from pathlib import Path
import math

import pandas as pd
import matplotlib.pyplot as plt

print("Setup OK")

## 1. Where is the file?

Before reading data, Python needs to know **where** the file is. A *path* is just the address of a file on disk. Our data lives in a folder called `data`, so the file we want is:

```text
data/tensile_lab_raw.csv
```

A CSV ("comma-separated values") file is just a plain-text table: one line per row, values separated by commas.

We build paths with `Path` and join the folder and file name with `/`.

In [ ]:
# The data folder, and the file we want to read
DATA_DIR = Path("data")
raw_file = DATA_DIR / "tensile_lab_raw.csv"

print("We will read:", raw_file)

### 🔧 A small helper (plumbing — you can skip the details)

In normal work you would simply write:

```python
df = pd.read_csv("data/tensile_lab_raw.csv")
```

That one line is all you really need. The helper in the next cell does exactly that, but it also tries a couple of other locations and, as a last resort, downloads the file from the course website — so the lesson works no matter how the files were uploaded, **especially inside the browser (JupyterLite)**.

You do **not** need to understand that cell. Just run it and move on.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# 🔧 PLUMBING — you do NOT need to understand this cell.
# In your own work you would simply write:
#       df = pd.read_csv("data/tensile_lab_raw.csv")
# This version just tries a few locations so the file always loads,
# including inside the browser (JupyterLite). Run it and move on.
# ──────────────────────────────────────────────────────────────────────

def load_table(name):
    # The normal way: look in the data folder, then the current folder.
    for folder in ("data", "."):
        try:
            return pd.read_csv(Path(folder) / name)
        except Exception:
            pass
    # Last resort (browser only): download the file from the course website.
    # This part is purely technical — not something you need for your own work.
    import js
    from pyodide.http import open_url
    base = str(js.location.href).split("/extensions/")[0]
    return pd.read_csv(open_url(f"{base}/files/data/{name}"))


print("Helper ready. (You can ignore how it works.)")

## 2. Read the file into a table

`pd.read_csv()` reads a CSV file and returns a **DataFrame** — pandas' word for a table (rows, columns, column names, values). We almost always store it in a variable called `df`.

Let's open the file the lab sent us and peek at the first rows with `.head()`.

*(We call the little helper above, but under the hood it is just `pd.read_csv`.)*

In [ ]:
df = load_table("tensile_lab_raw.csv")   # same as pd.read_csv("data/tensile_lab_raw.csv")

# The first thing to do after reading a file: look at the top rows.
df.head()

## 3. First look: what did the lab actually send?

Before any calculation, always *look* at the data. A few quick questions:

- how big is the table (rows × columns)?
- what are the columns, and what kind of values do they hold?
- do the numbers look plausible?

These 30 seconds of looking save hours of confusion later.

In [ ]:
print("Shape of the table:")
print(df.shape)
print()

print("Column names:")
print(df.columns)
print()

print("Data types:")
print(df.dtypes)

In [ ]:
# A compact summary of the table
# It shows column names, non-missing values, and data types.
df.info()

In [ ]:
# Summary statistics for numerical columns
df.describe()

## Exercise 1 — Basic inspection

Use pandas commands to answer these questions:

1. How many rows are in the dataset?
2. How many columns are in the dataset?
3. What are the first 10 rows?
4. What are the last 5 rows?
5. Which columns contain numerical values?

Useful commands:

```python
df.shape
df.head(10)
df.tail()
df.dtypes
```

In [ ]:
# Exercise 1
# Write your commands below.

# 1. Number of rows and columns


# 2. First 10 rows


# 3. Last 5 rows


# 4. Data types

## 4. Selecting columns

A DataFrame usually contains more information than we need at a given moment.

We can select one column:

```python
df["force_N"]
```

or several columns:

```python
df[["sample_id", "force_N", "displacement_mm"]]
```

The double square brackets in the second example are important: we are passing a list of column names.

In [ ]:
# Select one column
force = df["force_N"]

print(type(force))
print(force.head())

In [ ]:
# Select several columns
small_table = df[["sample_id", "treatment", "force_N", "displacement_mm"]]

small_table.head()

## 5. Filtering rows

Very often, we want to select only some rows.

For example:

- only one sample;
- only one treatment;
- only measurements above a certain force;
- only valid measurements.

To filter rows, we write a condition.

Example:

```python
df["sample_id"] == "S01"
```

This gives a column of `True` / `False` values.

Then we use that condition to select rows:

```python
df[df["sample_id"] == "S01"]
```

Remember:

- `=` assigns a value;
- `==` checks equality.

In [ ]:
# Create a Boolean condition
condition = df["sample_id"] == "S01"

condition.head(10)

In [ ]:
# Use the condition to filter rows
sample_s01 = df[df["sample_id"] == "S01"]

sample_s01.head()

In [ ]:
# Filter by treatment
annealed = df[df["treatment"] == "annealed"]

print("Rows in original dataset:", len(df))
print("Rows for annealed samples:", len(annealed))

annealed.head()

### Multiple conditions

To combine conditions in pandas, use:

- `&` for **and**;
- `|` for **or**;
- `~` for **not**.

Each condition should be placed inside parentheses.

Example:

```python
(df["sample_id"] == "S01") & (df["force_N"] > 100)
```

In [ ]:
# Example: rows for sample S01 where force is greater than 100 N
subset = df[(df["sample_id"] == "S01") & (df["force_N"] > 100)]

subset.head()

## Exercise 2 — Filtering

Create three filtered tables:

1. all rows for sample `S03`;
2. all rows where `force_N` is greater than 500;
3. all rows for quenched samples where `displacement_mm` is greater than 0.1.

Then print the number of rows in each filtered table.

In [ ]:
# Exercise 2
# Write your code below.

# 1. Rows for sample S03


# 2. Rows where force_N > 500


# 3. Rows for quenched samples where displacement_mm > 0.1

## 6. From force to strength (calculated columns)

The machine only records **force** and **displacement**. But "strength" is **stress** (force spread over the cross-section), and stretch is **strain**. So we compute them from what we have:

```text
area_mm2   = π × (diameter_mm / 2)²
stress_MPa = force_N / area_mm2
strain     = displacement_mm / gauge_length_mm
```

Handy unit fact: **1 N/mm² = 1 MPa**, so force in newtons ÷ area in mm² gives stress directly in MPa.

In pandas we create a whole new column in one line — the calculation runs on every row at once.

In [ ]:
# Create calculated columns

df["area_mm2"] = math.pi * (df["diameter_mm"] / 2) ** 2
df["stress_MPa"] = df["force_N"] / df["area_mm2"]
df["strain"] = df["displacement_mm"] / df["gauge_length_mm"]

# Inspect the new columns
df[["sample_id", "force_N", "displacement_mm", "area_mm2", "stress_MPa", "strain"]].head()

In [ ]:
# Check ranges of the new quantities
print("Stress range [MPa]:")
print(df["stress_MPa"].min(), "to", df["stress_MPa"].max())
print()

print("Strain range [-]:")
print(df["strain"].min(), "to", df["strain"].max())

## Exercise 3 — Add another calculated column

Create a new column called `strain_percent`.

It should be equal to:

```text
strain_percent = strain × 100
```

Then display these columns:

```text
sample_id, strain, strain_percent
```

for the first 10 rows.

In [ ]:
# Exercise 3
# Write your code below.

## 7. Find the glitches (data-quality checks)

Raw machine exports are rarely perfect — sensors hiccup and readings drop out. Before trusting any result, we look for obvious nonsense:

- **missing** values (`NaN`);
- **negative** forces or displacements (physically impossible here);
- unexpected sample names or treatment labels.

Let's see what this file is hiding.

In [ ]:
# Count missing values in each column
df.isna().sum()

In [ ]:
# Check unique samples and treatments
print("Sample IDs:")
print(df["sample_id"].unique())
print()

print("Treatments:")
print(df["treatment"].unique())

In [ ]:
# Check for negative values in quantities that should not be negative
negative_force_rows = df[df["force_N"] < 0]
negative_displacement_rows = df[df["displacement_mm"] < 0]

print("Rows with negative force:", len(negative_force_rows))
print("Rows with negative displacement:", len(negative_displacement_rows))

### From "raw" to "clean": removing bad rows

The checks above are not just for show — this raw dataset really does contain a few bad measurements:

- some **missing** force readings (`NaN`);
- a few **negative** forces and one negative displacement (sensor glitches).

A negative or missing force makes no physical sense, so we drop those rows to build a **clean** table. We keep the original `df` untouched and create a new table called `df_clean`.

In [ ]:
# Build a clean table by removing physically impossible rows
bad_rows = (
    df["force_N"].isna()
    | (df["force_N"] < 0)
    | (df["displacement_mm"] < 0)
)

df_clean = df[~bad_rows].copy()

print("Rows before cleaning:", len(df))
print("Rows removed:        ", int(bad_rows.sum()))
print("Rows after cleaning: ", len(df_clean))

## Exercise 4 — Sanity checks

Answer these questions with code:

1. How many different samples are in the dataset?
2. How many different treatments are in the dataset?
3. How many rows are present for each sample?
4. What is the maximum force measured for each sample?

Useful methods:

```python
.unique()
.nunique()
.value_counts()
.groupby()
.max()
```

In [ ]:
# Exercise 4
# Write your code below.

# 1. Number of different samples


# 2. Number of different treatments


# 3. Number of rows for each sample


# 4. Maximum force for each sample

## 8. One number per specimen — and the answer

We now have a clean table with ~120 measurement points per specimen. To compare specimens we need **one summary row each** — for example each specimen's **maximum stress**, which is its strength.

`groupby("sample_id")` splits the table into one group per specimen; `.agg(...)` then computes a value for each group.

```text
many rows per specimen → one summary row per specimen
```

In [ ]:
# For each specimen: its treatment, its peak stress (= strength) and peak strain
summary_by_sample = (
    df_clean
    .groupby("sample_id")
    .agg(
        treatment=("treatment", "first"),
        max_stress_MPa=("stress_MPa", "max"),
        max_strain=("strain", "max"),
    )
    .reset_index()
)

# Sort so the strongest specimen is on top
summary_by_sample = summary_by_sample.sort_values("max_stress_MPa", ascending=False)
summary_by_sample

### So, which treatment wins?

Let's average the peak stress *within each treatment* and sort from strongest to weakest.

In [ ]:
# Average peak stress for each treatment, strongest first
summary_by_treatment = (
    summary_by_sample
    .groupby("treatment")["max_stress_MPa"]
    .mean()
    .sort_values(ascending=False)
)

summary_by_treatment

> 🎯 **The answer.** The **quenched** specimen (S03) reaches the highest stress (~516 MPa), while the **annealed** ones are the softest (~355–367 MPa). So, for this dataset, **quenching gives the strongest material**.
>
> We started from a raw, slightly broken machine export and turned it into a one-line conclusion — that is exactly what this whole workflow is for.

## 9. Save the results

Colleagues, reports, or other notebooks should be able to use our results without redoing every step. So we save three files:

1. the full **cleaned** table;
2. the **summary by specimen**;
3. the **summary by treatment**.

We put them in a `results` folder, which we create on the fly.

In [ ]:
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)   # create the folder if it does not exist yet

# index=False: don't write the row-number index as an extra column
df_clean.to_csv(RESULTS_DIR / "tensile_clean.csv", index=False)
summary_by_sample.to_csv(RESULTS_DIR / "tensile_summary_by_sample.csv", index=False)
summary_by_treatment.to_csv(RESULTS_DIR / "tensile_summary_by_treatment.csv")

print("Saved 3 files into the 'results' folder.")

In [ ]:
# Read one of them back to confirm it was written correctly
pd.read_csv(RESULTS_DIR / "tensile_clean.csv").head()

## 10. A quick look at a curve

Numbers in a table are abstract. A quick plot tells us at a glance whether the cleaned data look sensible. Here is the stress–strain curve for one specimen.

In [ ]:
sample_id = "S01"
sample = df_clean[df_clean["sample_id"] == sample_id]

plt.figure(figsize=(6, 4))
plt.plot(sample["strain"], sample["stress_MPa"], marker="o", markersize=3)
plt.xlabel("Strain [-]")
plt.ylabel("Stress [MPa]")
plt.title(f"Stress-strain curve for sample {sample_id}")
plt.grid(True)
plt.show()

## 11. Mini-project — Clean and summarize a dataset

Use what we learned today to build a small analysis workflow.

### Task

Starting from `df`, produce:

1. a cleaned table with calculated columns;
2. a summary table by sample;
3. a summary table by treatment;
4. one simple plot for a chosen sample;
5. saved CSV files in the `results` folder.

### Minimum requirements

Your notebook should contain:

- a title;
- a short description of the dataset;
- data loading;
- data inspection;
- calculated columns;
- basic sanity checks;
- saved outputs.

### Optional extension

Repeat the same type of workflow for `hardness_positions.csv`.

In [ ]:
# Mini-project — starter scaffold
# Fill in each step. You can reuse code from the sections above.

# 1. Load the data (it is already in df / df_clean above; you may also reload it here)


# 2. Make sure the calculated columns exist (area_mm2, stress_MPa, strain)


# 3. Build a summary table by sample (groupby + agg)


# 4. Build a summary table by treatment


# 5. Make one stress-strain plot for a sample of your choice


# 6. Save your summary tables into the results folder with to_csv(...)

## Optional extension — Hardness data

The folder also contains a smaller dataset:

```text
data/hardness_positions.csv
```

This dataset contains repeated hardness measurements at several positions on the same samples.

Columns:

- `sample_id`;
- `treatment`;
- `position_id`;
- `hardness_HV`;
- `is_valid` (some measurements were flagged as invalid).

Possible tasks:

1. read the file;
2. keep only rows where `is_valid == True`;
3. calculate mean and standard deviation of hardness for each sample;
4. calculate mean hardness for each treatment;
5. save a summary CSV.

In [ ]:
# Optional extension
# Read hardness_positions.csv and summarize the valid measurements.

hardness_file = DATA_DIR / "hardness_positions.csv"

# Your code here

## Optional demonstration — Reading metadata from JSON

Not all useful information is stored in CSV files.

Sometimes metadata are stored in JSON files.

JSON is useful for structured information such as:

- instrument name;
- operator;
- sample preparation details;
- project information;
- analysis parameters.

This is optional, but it is good to know that Python can read these files too.

In [ ]:
import json

# (The try/except is just plumbing so the file loads in the browser too — ignore it.)
try:
    f = open(DATA_DIR / "sample_metadata.json")
except OSError:
    f = open("sample_metadata.json")

with f:
    metadata = json.load(f)

metadata

# End of Lesson 3

## What we did

Starting from a raw machine export, we:

- found and read a CSV file into a DataFrame;
- looked at it (`head`, `info`, `describe`, `dtypes`);
- selected columns and filtered rows;
- computed stress and strain;
- found and removed the bad measurements (the **cleaning** step);
- summarized one number per specimen and per treatment;
- saved the clean results — and answered the question: **quenching gives the strongest material**.

## The takeaway

Reading a file is only the start. The real workflow is:

```text
read → inspect → calculate → check → clean → summarize → save
```

In the next lesson, we focus on **plotting** and reading more of the science behind these curves.